Environment setting

In [ ]:
import gdown

file_id = "1Uu0PH3VgX5iR8UsCYt0y1S3lm9vhHLEv"
output = "packageset.tar"
gdown.download(id=file_id, output=output, quiet=False)

In [ ]:
!tar -xvf packageset.tar

In [ ]:
!pip install -q rpy2
%load_ext rpy2.ipython

In [ ]:
%%R

.libPaths("/content/content/drive/MyDrive/R_libs")

Loaindg necessary library and data

In [ ]:
#1-1. Loading necessary packages and practice data (< 1min)
%%R
library(Seurat)
library(scDblFinder)
library(SingleCellExperiment)
library(dplyr)
library(ggplot2)
library(patchwork)
library(EnhancedVolcano)
library(msigdbr)
library(clusterProfiler)

In [ ]:
#1-1. Loading practice data (< 1min)
%%R
setwd("/content/content/drive/MyDrive/R_libs")
hs09 <- readRDS("hs09.rds")
hs12 <- readRDS("hs12.rds")
hs13 <- readRDS("hs13.rds")
hs254 <- readRDS("hs254.rds")
hs255 <- readRDS("hs255.rds")
hs266 <- readRDS("hs266.rds")

Removing doublets

In [ ]:
#2-1. Convert your Seurat objects to SingleCellExperiment (< 1min)
%%R
sce09 <- as.SingleCellExperiment(hs09, assay = "RNA")
sce12 <- as.SingleCellExperiment(hs12, assay = "RNA")
sce13 <- as.SingleCellExperiment(hs13, assay = "RNA")
sce254 <- as.SingleCellExperiment(hs254, assay = "RNA")
sce255 <- as.SingleCellExperiment(hs255, assay = "RNA")
sce266 <- as.SingleCellExperiment(hs266, assay = "RNA")

In [ ]:
#2-2. Run scDblFinder (~10min)
%%R
sce09 <- scDblFinder(sce09)
sce12 <- scDblFinder(sce12)
sce13 <- scDblFinder(sce13)
sce254 <- scDblFinder(sce254)
sce255 <- scDblFinder(sce255)
sce266 <- scDblFinder(sce266)


In [ ]:
#2-3. Add the results back to your Seurat object (< 1min)
%%R
hs09$doublet_score <- sce09$scDblFinder.score
hs09$is_doublet <- sce09$scDblFinder.class

hs12$doublet_score <- sce12$scDblFinder.score
hs12$is_doublet <- sce12$scDblFinder.class

hs13$doublet_score <- sce13$scDblFinder.score
hs13$is_doublet <- sce13$scDblFinder.class

hs254$doublet_score <- sce254$scDblFinder.score
hs254$is_doublet <- sce254$scDblFinder.class

hs255$doublet_score <- sce255$scDblFinder.score
hs255$is_doublet <- sce255$scDblFinder.class

hs266$doublet_score <- sce266$scDblFinder.score
hs266$is_doublet <- sce266$scDblFinder.class

In [ ]:
#2-4. Check how many cells were detected as doublets (< 1min)
%%R
print("hs09")
print(table(hs09$is_doublet))
print("hs12")
print(table(hs12$is_doublet))
print("hs13")
print(table(hs13$is_doublet))
print("hs254")
print(table(hs254$is_doublet))
print("hs255")
print(table(hs255$is_doublet))
print("hs266")
print(table(hs266$is_doublet))

In [ ]:
#2-5. Filtering doublet cells (< 1min)
%%R
hs09 <- subset(hs09, subset = is_doublet == "singlet")
hs12 <- subset(hs12, subset = is_doublet == "singlet")
hs13 <- subset(hs13, subset = is_doublet == "singlet")
hs254 <- subset(hs254, subset = is_doublet == "singlet")
hs255 <- subset(hs255, subset = is_doublet == "singlet")
hs266 <- subset(hs266, subset = is_doublet == "singlet")

In [ ]:
#2-6. Merge separate datas into one for convenience in further analysis (< 1min)
%%R
data <- merge(
  x = hs09,
  y = c(hs12, hs13, hs254, hs255, hs266),
  add.cell.ids = c("Hs09", "Hs12", "Hs13", "Hs254", "Hs255", "Hs266"),
)

set.seed(42)

cells_to_keep <- sample(colnames(data), size = 6000)
data <- subset(data, cells = cells_to_keep)

Quality control

In [ ]:
#3-1. Take a look at nFeature, nCount, percent.mt metrics within our data (< 1min)
%%R
VlnPlot(data, group.by = 'sample',feature = c("nFeature_RNA","nCount_RNA","percent.mt"),ncol=3)

In [ ]:
#3-2. Filtering cells with high mitochondrial percentage (< 1min)
%%R
data <- subset(data, subset = percent.mt < 20 & nCount_RNA >= 200)

Normalization

In [ ]:
#4. Normalization (< 1min)
%%R
data <- NormalizeData(data, normalization.method = "LogNormalize", scale.factor = 10000)
gc()

Feature selection

In [ ]:
#5-1. Finding variable features (< 1min)
%%R
data <- FindVariableFeatures(data, selection.method = "vst", nfeatures = 2000)
gc()

In [ ]:
#5-2.Checking variable features (< 1min)
%%R
top10 <- head(VariableFeatures(data), 10)
plot1 <- VariableFeaturePlot(data)
plot2 <- LabelPoints(plot = plot1, points = top10, repel = TRUE)
plot2

Scaling

In [ ]:
#6. Scaling (< 1min)
%%R
data <- ScaleData(data)

Linear dimension reduction(PCA)

In [ ]:
#7-1. Run PCA (< 1min)
%%R
set.seed(42)
data <- RunPCA(data, verbose = FALSE)

In [ ]:
#7-2. Examine and visualize PCA results (< 1min)
%%R
print(data[["pca"]], dims = 1:5, nfeatures = 5)
DimHeatmap(data, dims = 1:15, cells = 500, balanced = TRUE)

In [ ]:
#7-3. Elbow plot (Determining # of PCs to use for further analysis) (< 1min)
%%R
stdev <- data[["pca"]]@stdev
pct <- stdev / sum(stdev) * 100
cum_pct <- cumsum(pct)
pcs <- 1:length(stdev)
plot_df <- data.frame(PC = pcs, stdev = stdev, pct = pct, cum_pct = cum_pct)

pc_90 <- which(cum_pct >= 90)[1]
pc_01 <- which(diff(pct) > -0.1)[1]

p_main <- ggplot(plot_df, aes(x = PC, y = stdev)) +
  geom_point(size = 1.5) +
  geom_vline(xintercept = pc_90, color = "royalblue", linetype = "solid") +
  geom_vline(xintercept = pc_01, color = "red", linetype = "dashed") +
  theme_classic() +
  labs(title = "Method for selecting the appropriate number of PCs:\n1) 90% variance (Blue line)\n2) <0.1% drop (Red line)",
       y = "Standard Deviation")

In [ ]:
#7-4. Create the Cumulative Contribution Subplots (< 1min)
# Subplot 1: 90% Threshold
%%R
p1 <- ggplot(plot_df, aes(x = cum_pct, y = stdev)) +
  geom_point(aes(color = PC <= pc_90), size = 1) +
  geom_text(aes(label = PC, color = PC <= pc_90), vjust = -1, size = 2, alpha = 0.5) +
  scale_color_manual(values = c("TRUE" = "cyan", "FALSE" = "tomato")) +
  theme_bw() +
  labs(title = paste0("PC = ", pc_90, " <- 90% reached"),
       x = "Cumulative % Contribution", y = "SD (%)") +
  theme(legend.position = "none")
  p1

In [ ]:
#7-5. Subplot 2: 0.1% Drop Threshold (< 1min)
%%R
p2 <- ggplot(plot_df, aes(x = cum_pct, y = stdev)) +
  geom_point(aes(color = PC <= pc_01), size = 1) +
  geom_text(aes(label = PC, color = PC <= pc_01), vjust = -1, size = 2, alpha = 0.5) +
  scale_color_manual(values = c("TRUE" = "cyan", "FALSE" = "tomato")) +
  theme_bw() +
  labs(title = paste0("PC = ", pc_01, " <- <0.1% drop"),
       x = "Cumulative % Contribution", y = "SD (%)") +
  theme(legend.position = "none")
  p2

Clustering

In [ ]:
#8. Clustering the cell (1min)
%%R
data <- FindNeighbors(data, dims = 1:10,reduction='pca',seed.use=42)
data <- FindClusters(data, resolution = 0.1,random.seed=42)

Non-linear dimesionality reduction(UMAP)

In [ ]:
#9-1. Run UMAP (1min)
%%R
data <- RunUMAP(data, dims = 1:10)


In [ ]:
#9-2. Examine our clusters (< 1min)
%%R
DimPlot(data,reduction="umap",label = TRUE)

Annotation

In [ ]:
#10-1. Checking for canonical markers (< 1min)
%%R
FeaturePlot(data, features = c("ADIPOQ"),label=TRUE) #Adipocyte cell marker

In [ ]:
#10-2. Checking for canonical markers (< 1min)
%%R
FeaturePlot(data, features = c("PTPRC"),label=TRUE) #Immune cell marker

In [ ]:
#10-3. Checking canonical marker expression profile in our clusters (< 1min)
%%R
VlnPlot(data, features=c('ADIPOQ','PTPRC','PROX1','STEAP4','JAM2'), group.by='seurat_clusters', pt.size=0) & theme(aspect.ratio=1)


10-4. Finding differentially expressed genes cluster2(presumably adipocyte) vs other clusters

This step was performed ahead of the exercise due to time limit. Please refer to the cluster2marker file.

%%R data <- JoinLayers(data)

cluster2_markers <- FindMarkers(data, ident.1 = 2) cluster2_markers %>% dplyr::filter(avg_log2FC > 1)

In [ ]:
#10-5. Annotation (< 1min)
%%R
new.cluster.ids <- c("Fibroblast", "ASPC", "Adipose cells", "Immunce cells", "Endothelial cells", "Immune cells",
                     "LEC", "Pericyte")
names(new.cluster.ids) <- levels(data)
data <- RenameIdents(data, new.cluster.ids)
data$cell_type <- Idents(data)

Differential abundance analysis between 'normal' vs 'obese'

In [ ]:
#11-1. Create a frequency table of cell type vs condition and calculate percentages for each condition (< 1min)
%%R
data$condition <- ifelse(data$bmi <=25,'normal','obese')
counts <- as.data.frame(table(data$cell_type, data$condition))
colnames(counts) <- c("Cell_type", "Condition", "Freq")

counts <- counts %>%
  group_by(Condition) %>%
  mutate(Percentage = (Freq / sum(Freq)) * 100)

In [ ]:
#11-2. Draw the Stacked Barplot (< 1min)
%%R
ggplot(counts, aes(x = Condition, y = Percentage, fill = Cell_type)) +
  geom_bar(stat = "identity", color = "white") +
  theme_minimal() +
  labs(title = "Cell Composition by BMI Condition",
       x = "Condition (BMI Status)",
       y = "Percentage of Total Cells (%)",
       fill = "Cell Type") +
  scale_fill_brewer(palette = "Set3") + # Change palette as needed
  theme(axis.text = element_text(size = 12),
        legend.title = element_text(face = "bold"))

Differential expression analysis(gene expression difference in adipose cells of normal vs obese)

In [ ]:
#12-1. Run differential expression analysis (2min)
%%R
adipocyte <- subset(data,subset=cell_type=='Adipose cells')
adipocyte <- JoinLayers(adipocyte)
Idents(adipocyte) <- "condition"
adipocyte_de <- FindMarkers(adipocyte,
                                    ident.1 = "obese",
                                    ident.2 = "normal")

In [ ]:
#12-2. Visualization via volcano plot (< 1min)
%%R
EnhancedVolcano(adipocyte_de,
                lab = rownames(adipocyte_de),
                x = 'avg_log2FC',
                y = 'p_val_adj',
                drawConnectors = TRUE)

Pathway analysis

In [ ]:
#13-1. Create a ranking metric and sorting by descending order (< 1min)
%%R
adipocyte_de$ranking_metric <- sign(adipocyte_de$avg_log2FC) * (-log10(adipocyte_de$p_val_adj + 1e-300))
gene_list <- adipocyte_de$ranking_metric
names(gene_list) <- rownames(adipocyte_de)
gene_list <- sort(gene_list, decreasing = TRUE)

In [ ]:
#13-2. Download Hallmark gene sets for Humans (1min)
%%R
h_df <- msigdbr(species = "Homo sapiens", category = "H") %>%
  dplyr::select(gs_name, gene_symbol)

In [ ]:
#13-3. Run GSEA(<1min)
%%R
gsea_res <- GSEA(gene_list, TERM2GENE = h_df, pvalueCutoff = 0.2)

In [ ]:
#13-4. Visualize GSEA result
%%R
dotplot(gsea_res, showCategory = 10, split = ".sign") +
  facet_grid(.~.sign) +
  theme_minimal() +
  labs(title = "GSEA: Obese vs. Normal Adipocytes")

Thank you for your time